In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("../data/raw/ApexPlanet_DataAnalytics_Dataset.xlsx")

print(df.shape)
df.head()

(1000, 12)


,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales
0,ORD100002,2025-02-25,CUST5529,Customer_227,30.0,Female,Bengaluru,Rice,Grocery,7,2829.77,19808.39
1,ORD100003,2025-10-14,CUST3127,Customer_182,63.0,Male,Bengaluru,Book,Education,5,27906.16,139530.80
2,ORD100004,2025-05-13,CUST8887,Customer_487,62.0,Female,Bengaluru,Book,Education,8,37491.06,299928.48
3,ORD100005,2025-12-02,CUST2515,Customer_470,65.0,Female,Kolkata,Mobile,Electronics,9,28541.36,256872.24
4,ORD100006,2025-11-20,CUST4796,Customer_380,44.0,Male,Bengaluru,Rice,Grocery,10,14036.59,140365.90


In [3]:
df_clean = df.copy()

### Convert Order_Date

In [4]:
df_clean["Order_Date"].dtype

dtype('O')

In [5]:
df_clean["Order_Date"] = pd.to_datetime(df_clean["Order_Date"])

In [6]:
df_clean["Order_Date"].dtype

dtype('<M8[ns]')

### Handle Missing Age

In [7]:
df_clean["Age"].isnull().sum()

np.int64(20)

In [8]:
median_age = df_clean["Age"].median()

df_clean["Age"] = df_clean["Age"].fillna(median_age)

In [9]:
df_clean["Age"].isnull().sum()

np.int64(0)

### Handle Missing City

In [10]:
df_clean["City"].isnull().sum()

np.int64(13)

In [11]:
df_clean["City"].mode()

0    Patna
Name: City, dtype: object

In [12]:
mode_city = df_clean["City"].mode()[0]

df_clean["City"] = df_clean["City"].fillna(mode_city)

In [13]:
df_clean["City"].isnull().sum()

np.int64(0)

### Validate Missing Values

In [14]:
df_clean.isnull().sum()

Order_ID         0
Order_Date       0
Customer_ID      0
Customer_Name    0
Age              0
Gender           0
City             0
Product          0
Category         0
Quantity         0
Unit_Price       0
Total_Sales      0
dtype: int64

### Validate Total Sales Formula

In [16]:
df_clean["Calculated_Sales"] = (
    df_clean["Quantity"] *
    df_clean["Unit_Price"]
)

In [25]:
np.isclose(
    df_clean["Calculated_Sales"],
    df_clean["Total_Sales"]
).all()

np.True_

In [26]:
print(
    "Sales Validation:",
    np.isclose(
        df_clean["Calculated_Sales"],
        df_clean["Total_Sales"]
    ).all()
)

Sales Validation: True


### Create Age_Group Feature

In [19]:
df_clean["Age_Group"] = pd.cut(
    df_clean["Age"],
    bins=[0,30,45,60,100],
    labels=[
        "Young",
        "Adult",
        "Middle Age",
        "Senior"
    ]
)

In [20]:
df_clean["Age_Group"].value_counts()

Age_Group
Adult         332
Middle Age    295
Young         268
Senior        105
Name: count, dtype: int64

### Outlier Analysis

In [22]:
# Age
Q1 = df_clean["Age"].quantile(0.25)
Q3 = df_clean["Age"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df_clean[
    (df_clean["Age"] < lower) |
    (df_clean["Age"] > upper)
]

print(len(outliers))

0


In [23]:
# Total Sales

Q1 = df_clean["Total_Sales"].quantile(0.25)
Q3 = df_clean["Total_Sales"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

sales_outliers = df_clean[
    (df_clean["Total_Sales"] < lower) |
    (df_clean["Total_Sales"] > upper)
]

print(len(sales_outliers))

19


In [24]:
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])

df_clean.info()

Rows: 1000
Columns: 14
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Order_ID          1000 non-null   object        
 1   Order_Date        1000 non-null   datetime64[ns]
 2   Customer_ID       1000 non-null   object        
 3   Customer_Name     1000 non-null   object        
 4   Age               1000 non-null   float64       
 5   Gender            1000 non-null   object        
 6   City              1000 non-null   object        
 7   Product           1000 non-null   object        
 8   Category          1000 non-null   object        
 9   Quantity          1000 non-null   int64         
 10  Unit_Price        1000 non-null   float64       
 11  Total_Sales       1000 non-null   float64       
 12  Calculated_Sales  1000 non-null   float64       
 13  Age_Group         1000 non-null   category      
dtypes:

In [27]:
# Remove temporary validation column
df_clean.drop(columns=["Calculated_Sales"], inplace=True)

# Save cleaned dataset
df_clean.to_csv(
    "../data/processed/cleaned_dataset.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


# Cleaning Summary

## Cleaning Actions Performed

### Date Conversion
- Converted Order_Date from object datatype to datetime format.

### Missing Value Treatment
- Filled missing Age values using the median age.
- Filled missing City values using the mode city (Patna).

### Data Validation
- Verified Total_Sales using Quantity × Unit_Price.
- Validation differences were due to floating-point precision only.

### Feature Engineering
- Created a new Age_Group feature:
  - Young (0–30)
  - Adult (31–45)
  - Middle Age (46–60)
  - Senior (60+)

### Outlier Analysis
- No Age outliers detected.
- 19 Total_Sales outliers identified and retained as valid business transactions.

## Final Dataset Status

### Before Cleaning
- Missing Age values: 20
- Missing City values: 13
- Order_Date stored as object datatype

### After Cleaning
- Missing Age values: 0
- Missing City values: 0
- Order_Date converted to datetime
- Age_Group feature added

## Dataset Ready for Analysis
The dataset has been successfully cleaned, validated, and transformed into an analysis-ready format suitable for Exploratory Data Analysis (EDA), SQL-based business intelligence, dashboard development, and statistical analysis.